<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/SymmetryDial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 7.1 MB/s eta 0:00:00


In [ ]:

# ==============================================================================
#  PREAMBLE: IMPORTS AND SETUP
# ==============================================================================
import numpy as np
import time
import warnings
from scipy.linalg import expm
from scipy.special import rel_entr # For KL Divergence

# Qiskit Imports
import qiskit
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

# The powerful "Designer AI" engine
import cma

# Suppress benign warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Qiskit version: {qiskit.__version__}")
print("\n--- The AI Physicist: Discovering the Laws of a Toy Universe (Corrected Version) ---")

# ==============================================================================
#  PART 1: CREATE AND SIMULATE THE TOY UNIVERSE
# ==============================================================================
print("\n--- Part 1: Defining and Simulating the 'Toy Universe' ---")

N_QUBITS = 6
TIME_STEPS = 3
PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

# --- Define the "True Law of Physics" for our universe ---
true_coeffs = np.zeros(15)
true_coeffs[PAULI_BASIS_2Q.index('XX')] = 0.8
true_coeffs[PAULI_BASIS_2Q.index('YY')] = 0.8
true_coeffs[PAULI_BASIS_2Q.index('ZI')] = -0.3
print("  > The 'True Law of Physics' (generator H) has been defined.")

# Create the unitary gate U_phys = e^(-iH) from the true law
true_h_matrix = SparsePauliOp(PAULI_BASIS_2Q, coeffs=true_coeffs).to_matrix()
U_PHYSICS_GATE = expm(-1j * true_h_matrix)

# --- Generate the "Experimental Data" ---
def run_universe_simulation(u_gate, initial_state_int):
    """Evolves the universe for a few time steps."""
    qc = QuantumCircuit(N_QUBITS)
    qc.initialize(Statevector.from_int(initial_state_int, 2**N_QUBITS))
    for t in range(TIME_STEPS):
        for i in range(N_QUBITS - 1):
            qc.unitary(u_gate, [i, i+1], label=f"U_phys_t{t}")
    final_state = Statevector.from_instruction(qc)
    return final_state.probabilities()

print("  > Running the universe simulation to generate 'experimental data'...")
initial_state_int = int('110000', 2)
experimental_data_dist = run_universe_simulation(U_PHYSICS_GATE, initial_state_int)
print("  > 'Experimental data' (a probability distribution) has been collected.")

# ==============================================================================
#  PART 2: THE AI PHYSICIST TRIES TO DISCOVER THE LAW
# ==============================================================================
print("\n--- Part 2: The AI Physicist's Quest for Discovery ---")

# --- The AI's Fitness Function ---
def ai_physicist_fitness(trial_coeffs: np.ndarray) -> float:
    try:
        trial_h_matrix = SparsePauliOp(PAULI_BASIS_2Q, coeffs=trial_coeffs).to_matrix()
        u_trial = expm(-1j * trial_h_matrix)
        trial_data_dist = run_universe_simulation(u_trial, initial_state_int)
        kl_divergence = np.sum(rel_entr(experimental_data_dist + 1e-9, trial_data_dist + 1e-9))
        return kl_divergence
    except Exception:
        return 1e6

# --- Set up and run the "Master" CMA-ES Optimizer ---
print("  > Starting AI Physicist (CMA-ES) to search for the physical law...")

# --- THE FIX IS HERE ---
lower_bound = -1.0
upper_bound = 1.0
# Generate a random starting point that is guaranteed to be within the bounds.
x0 = lower_bound + (upper_bound - lower_bound) * np.random.rand(15)
# --- END OF FIX ---

sigma0 = 0.5
options = {'bounds': [lower_bound, upper_bound], 'maxfevals': 3000, 'verbose': -9}
master_es = cma.CMAEvolutionStrategy(x0, sigma0, options)

start_time = time.time()
master_es.optimize(ai_physicist_fitness)
end_time = time.time()
print(f"  > AI search complete in {end_time - start_time:.2f}s.")

# --- Retrieve the law discovered by the AI ---
discovered_coeffs = master_es.result.xbest
final_kl_divergence = master_es.result.fbest

# ==============================================================================
#  FINAL VERDICT: DID THE AI SUCCEED?
# ==============================================================================
print("\n--- FINAL VERDICT ---")
print(f"Final KL Divergence: {final_kl_divergence:.6f} (closer to 0 is better)")

print("\nComparison of True Law vs. AI's Discovered Law:")
print("-" * 55)
print(f"{'Pauli Term':<15} | {'True Coeff':<15} | {'Discovered Coeff':<20}")
print("-" * 55)

total_error = 0
for i in range(15):
    pauli_term = PAULI_BASIS_2Q[i]
    true_c = true_coeffs[i]
    disc_c = discovered_coeffs[i]
    total_error += (true_c - disc_c)**2
    if abs(true_c) > 0.1 or abs(disc_c) > 0.1:
        print(f"{pauli_term:<15} | {true_c:<15.3f} | {disc_c:<20.3f}")
print("-" * 55)

mean_squared_error = total_error / 15
print(f"Mean Squared Error: {mean_squared_error:.6f}")

print("\n--- Final Conclusion ---")
if mean_squared_error < 0.01:
    print(">>> SUCCESS! The AI Physicist has successfully reverse-engineered the laws of our toy universe! <<<")
    print("It correctly identified the dominant physical interactions and their strengths.")
else:
    print(">>> The mystery remains. The AI could not fully deduce the physical law. <<<")
    print("This could be due to the complexity of the landscape or needing more simulation data/time.")

Qiskit version: 2.1.0

--- The AI Physicist: Discovering the Laws of a Toy Universe (Corrected Version) ---

--- Part 1: Defining and Simulating the 'Toy Universe' ---
  > The 'True Law of Physics' (generator H) has been defined.
  > Running the universe simulation to generate 'experimental data'...
  > 'Experimental data' (a probability distribution) has been collected.

--- Part 2: The AI Physicist's Quest for Discovery ---
  > Starting AI Physicist (CMA-ES) to search for the physical law...
  > AI search complete in 96.20s.

--- FINAL VERDICT ---
Final KL Divergence: 0.015212 (closer to 0 is better)

Comparison of True Law vs. AI's Discovered Law:
-------------------------------------------------------
Pauli Term      | True Coeff      | Discovered Coeff    
-------------------------------------------------------
IX              | 0.000           | 0.425               
IY              | 0.000           | 0.281               
IZ              | 0.000           | -0.977              
X

In [ ]:

# ==============================================================================
#  PREAMBLE: IMPORTS AND SETUP
# ==============================================================================
import numpy as np
import time
import warnings
from scipy.linalg import expm
from scipy.special import rel_entr # For KL Divergence

# Qiskit Imports
import qiskit
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

# The powerful "Designer AI" engine
import cma

# Suppress benign warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Qiskit version: {qiskit.__version__}")
print("\n--- The AI Physicist (Definitive Version): Discovering Universal Laws ---")

# ==============================================================================
#  PART 1: CREATE AND SIMULATE THE TOY UNIVERSE
# ==============================================================================
print("\n--- Part 1: Defining and Simulating the 'Toy Universe' ---")

N_QUBITS = 6
TIME_STEPS = 3
PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

# --- Define the "True Law of Physics" for our universe ---
true_coeffs = np.zeros(15)
true_coeffs[PAULI_BASIS_2Q.index('XX')] = 0.8
true_coeffs[PAULI_BASIS_2Q.index('YY')] = 0.8
true_coeffs[PAULI_BASIS_2Q.index('ZI')] = -0.3
print("  > The 'True Law of Physics' (generator H) has been defined.")

# Create the unitary gate U_phys = e^(-iH) from the true law
true_h_matrix = SparsePauliOp(PAULI_BASIS_2Q, coeffs=true_coeffs).to_matrix()
U_PHYSICS_GATE = expm(-1j * true_h_matrix)

# --- Generate the "Experimental Data" from MULTIPLE initial states ---
def run_universe_simulation(u_gate, initial_state_int):
    """Evolves the universe for a few time steps."""
    qc = QuantumCircuit(N_QUBITS)
    qc.initialize(Statevector.from_int(initial_state_int, 2**N_QUBITS))
    for t in range(TIME_STEPS):
        for i in range(N_QUBITS - 1):
            qc.unitary(u_gate, [i, i+1], label=f"U_phys_t{t}")
    final_state = Statevector.from_instruction(qc)
    return final_state.probabilities()

# --- THE FIX IS HERE: Create a richer dataset ---
# We use a set of orthogonal basis states to "probe" the universe's laws
BASIS_STATES = ['000000', '100000', '010000', '001000']
experimental_data_set = {}
print("  > Running multiple experiments to generate a rich dataset...")
for state_str in BASIS_STATES:
    initial_state_int = int(state_str, 2)
    experimental_data_set[state_str] = run_universe_simulation(U_PHYSICS_GATE, initial_state_int)
print("  > 'Experimental data' collected from multiple starting conditions.")
# --- END OF FIX ---

# ==============================================================================
#  PART 2: THE AI PHYSICIST TRIES TO DISCOVER THE LAW
# ==============================================================================
print("\n--- Part 2: The AI Physicist's Quest for Discovery ---")

# --- The AI's Robust Fitness Function ---
def ai_physicist_fitness(trial_coeffs: np.ndarray) -> float:
    try:
        trial_h_matrix = SparsePauliOp(PAULI_BASIS_2Q, coeffs=trial_coeffs).to_matrix()
        u_trial = expm(-1j * trial_h_matrix)

        total_kl_divergence = 0
        # The AI must now succeed on ALL experiments
        for state_str, true_dist in experimental_data_set.items():
            initial_state_int = int(state_str, 2)
            trial_dist = run_universe_simulation(u_trial, initial_state_int)
            kl_div = np.sum(rel_entr(true_dist + 1e-9, trial_dist + 1e-9))
            total_kl_divergence += kl_div

        return total_kl_divergence

    except Exception:
        return 1e6

# --- Set up and run the "Master" CMA-ES Optimizer ---
print("  > Starting AI Physicist (CMA-ES) to search for the universal physical law...")
lower_bound = -1.0
upper_bound = 1.0
x0 = lower_bound + (upper_bound - lower_bound) * np.random.rand(15)
sigma0 = 0.5
options = {'bounds': [lower_bound, upper_bound], 'maxfevals': 5000, 'verbose': -9} # Increased budget
master_es = cma.CMAEvolutionStrategy(x0, sigma0, options)

start_time = time.time()
master_es.optimize(ai_physicist_fitness)
end_time = time.time()
print(f"  > AI search complete in {end_time - start_time:.2f}s.")

# --- Retrieve the law discovered by the AI ---
discovered_coeffs = master_es.result.xbest
final_total_kl = master_es.result.fbest

# ==============================================================================
#  FINAL VERDICT: DID THE AI SUCCEED?
# ==============================================================================
print("\n--- FINAL VERDICT ---")
print(f"Final Total KL Divergence: {final_total_kl:.6f} (closer to 0 is better)")

print("\nComparison of True Law vs. AI's Discovered Law:")
print("-" * 55)
print(f"{'Pauli Term':<15} | {'True Coeff':<15} | {'Discovered Coff':<20}")
print("-" * 55)

total_error = 0
for i in range(15):
    pauli_term = PAULI_BASIS_2Q[i]
    true_c = true_coeffs[i]
    disc_c = discovered_coeffs[i]
    total_error += (true_c - disc_c)**2
    if abs(true_c) > 0.01 or abs(disc_c) > 0.1: # Threshold to show important terms
        print(f"{pauli_term:<15} | {true_c:<15.3f} | {disc_c:<20.3f}")
print("-" * 55)

mean_squared_error = total_error / 15
print(f"Mean Squared Error: {mean_squared_error:.6f}")

print("\n--- Final Conclusion ---")
if mean_squared_error < 0.01:
    print(">>> SUCCESS! The AI Physicist has successfully reverse-engineered the laws of our toy universe! <<<")
    print("By providing diverse experimental data, we forced the AI to find the one true, universal law.")
else:
    print(">>> The mystery remains. The AI could not fully deduce the physical law. <<<")
    print("This indicates the optimization is very challenging, or more data/evaluations are needed.")

Qiskit version: 2.1.0

--- The AI Physicist (Definitive Version): Discovering Universal Laws ---

--- Part 1: Defining and Simulating the 'Toy Universe' ---
  > The 'True Law of Physics' (generator H) has been defined.
  > Running multiple experiments to generate a rich dataset...
  > 'Experimental data' collected from multiple starting conditions.

--- Part 2: The AI Physicist's Quest for Discovery ---
  > Starting AI Physicist (CMA-ES) to search for the universal physical law...
  > AI search complete in 264.10s.

--- FINAL VERDICT ---
Final Total KL Divergence: 0.000000 (closer to 0 is better)

Comparison of True Law vs. AI's Discovered Law:
-------------------------------------------------------
Pauli Term      | True Coeff      | Discovered Coff     
-------------------------------------------------------
IX              | 0.000           | 0.385               
IY              | 0.000           | -0.997              
IZ              | 0.000           | 0.713               
XI     

In [ ]:

# ==============================================================================
#  PREAMBLE: IMPORTS AND SETUP
# ==============================================================================
import numpy as np
import time
import warnings
from scipy.linalg import expm
from scipy.special import rel_entr

# Qiskit Imports
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

# The powerful "Designer AI" engine
import cma

# Suppress benign warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("\n--- The Symmetry Dial Experiment: Testing the P vs. NP Hypothesis (Failsafe Version) ---")

# ==============================================================================
#  CORE "AI PHYSICIST" ENGINE
# ==============================================================================
N_QUBITS = 4
TIME_STEPS = 2
PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']
BASIS_STATES = ['0000', '1000', '0100']

def run_universe_simulation(u_gate, initial_state_int):
    qc = QuantumCircuit(N_QUBITS)
    qc.initialize(Statevector.from_int(initial_state_int, 2**N_QUBITS))
    for t in range(TIME_STEPS):
        for i in range(N_QUBITS - 1):
            qc.unitary(u_gate, [i, i+1])
    return Statevector.from_instruction(qc).probabilities()

def run_ai_physicist(true_coeffs):
    """The main experimental pipeline for one universe."""

    true_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=true_coeffs).to_matrix()
    u_physics = expm(-1j * true_h)
    experimental_data_set = {
        s: run_universe_simulation(u_physics, int(s, 2)) for s in BASIS_STATES
    }

    # --- THE FAILSAFE FIX IS HERE: Manual Counter ---
    # We use a list so it can be modified by the inner function (pass-by-reference)
    eval_counter = [0]
    # --- END OF FAILSAFE FIX ---

    def fitness(trial_coeffs):
        # --- Manually increment the counter each time the function is called ---
        eval_counter[0] += 1
        try:
            trial_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=trial_coeffs).to_matrix()
            u_trial = expm(-1j * trial_h)
            total_kl = sum(
                np.sum(rel_entr(true_dist + 1e-9, run_universe_simulation(u_trial, int(s, 2)) + 1e-9))
                for s, true_dist in experimental_data_set.items()
            )
            return total_kl
        except Exception: return 1e6

    x0 = -1.0 + 2.0 * np.random.rand(15)
    # We set a large budget, but will break early on success.
    es = cma.CMAEvolutionStrategy(x0, 0.5, {'bounds': [-np.pi, np.pi], 'maxfevals': 10000, 'verbose': -9})

    start_time = time.time()
    while not es.stop():
        solutions = es.ask()
        es.tell(solutions, [fitness(s) for s in solutions])
        if es.result.fbest < 0.001:
            print("  > Success! Law discovered.")
            break

    end_time = time.time()

    # Return the value from our manual counter
    return eval_counter[0], end_time - start_time, es.result.fbest

# ==============================================================================
#  THE THREE UNIVERSES
# ==============================================================================
if __name__ == "__main__":

    results = {}

    # --- Universe 1: Ordered (P-like) ---
    print("\n--- Testing Universe 1: The 'Ordered' World (P-like Symmetry) ---")
    h_ordered_coeffs = np.zeros(15)
    h_ordered_coeffs[PAULI_BASIS_2Q.index('ZZ')] = np.pi / 2
    evals, duration, final_fitness = run_ai_physicist(h_ordered_coeffs)
    results["Ordered"] = {'evals': evals, 'time': duration, 'fitness': final_fitness}
    print(f"  > Time to Discover: {duration:.2f}s ({evals} evaluations)")

    # --- Universe 2: Structured (BQP-like) ---
    print("\n--- Testing Universe 2: The 'Structured' World (BQP-like Symmetry) ---")
    h_structured_coeffs = np.zeros(15)
    h_structured_coeffs[PAULI_BASIS_2Q.index('XX')] = 0.5
    h_structured_coeffs[PAULI_BASIS_2Q.index('YY')] = 0.5
    h_structured_coeffs[PAULI_BASIS_2Q.index('ZZ')] = 0.5
    evals, duration, final_fitness = run_ai_physicist(h_structured_coeffs)
    results["Structured"] = {'evals': evals, 'time': duration, 'fitness': final_fitness}
    print(f"  > Time to Discover: {duration:.2f}s ({evals} evaluations)")

    # --- Universe 3: Chaotic (NP-Hard-like) ---
    print("\n--- Testing Universe 3: The 'Chaotic' World (No Symmetry) ---")
    np.random.seed(42)
    h_chaotic_coeffs = -np.pi + 2 * np.pi * np.random.rand(15)
    evals, duration, final_fitness = run_ai_physicist(h_chaotic_coeffs)
    results["Chaotic"] = {'evals': evals, 'time': duration, 'fitness': final_fitness}
    print(f"  > Time to Discover: {duration:.2f}s ({evals} evaluations)")


    # ==============================================================================
    #  FINAL VERDICT
    # ==============================================================================
    print("\n\n--- FINAL VERDICT: THE SYMMETRY DIAL ---")
    print("-" * 60)
    print(f"{'Universe Type':<20} | {'Evaluations to Solve':<25} | {'Final Fitness':<15}")
    print("-" * 60)
    for name, res in results.items():
        print(f"{name:<20} | {res['evals']:<25} | {res['fitness']:.6f}")
    print("-" * 60)

    print("\n--- Conclusion ---")
    ordered_evals = results["Ordered"]['evals']
    structured_evals = results["Structured"]['evals']
    chaotic_evals = results["Chaotic"]['evals']

    # Check if the chaotic universe failed to converge
    chaotic_failed = results["Chaotic"]['fitness'] > 0.001

    if ordered_evals < structured_evals and (structured_evals < chaotic_evals or chaotic_failed):
        print(">>> HYPOTHESIS CONFIRMED! <<<")
        print("The AI found the law for the 'Ordered' universe fastest, the 'Structured' universe slower,")
        print("and struggled immensely with the 'Chaotic' universe, often failing to find the law within the budget.")
        print("This provides strong experimental evidence that computational difficulty is directly linked")
        print("to the degree of exploitable symmetry in a problem's structure.")
    else:
        print(">>> RESULT IS INCONCLUSIVE OR CONTRADICTS HYPOTHESIS <<<")
        print("The search difficulty did not scale with the apparent symmetry of the problem,")
        print("suggesting the fitness landscape is more complex than anticipated.")


--- The Symmetry Dial Experiment: Testing the P vs. NP Hypothesis (Failsafe Version) ---

--- Testing Universe 1: The 'Ordered' World (P-like Symmetry) ---
  > Success! Law discovered.
  > Time to Discover: 15.79s (1452 evaluations)

--- Testing Universe 2: The 'Structured' World (BQP-like Symmetry) ---
  > Success! Law discovered.
  > Time to Discover: 17.90s (1716 evaluations)

--- Testing Universe 3: The 'Chaotic' World (No Symmetry) ---
  > Time to Discover: 80.88s (8448 evaluations)


--- FINAL VERDICT: THE SYMMETRY DIAL ---
------------------------------------------------------------
Universe Type        | Evaluations to Solve      | Final Fitness  
------------------------------------------------------------
Ordered              | 1452                      | 0.000976
Structured           | 1716                      | 0.000783
Chaotic              | 8448                      | 0.335879
------------------------------------------------------------

--- Conclusion ---
>>> HYPOTHESI